# DS4 Network Theory (NeT) -- Stage 4 Step 1

**Notebook:** `54_ds4_network_models.ipynb`  
**Model family:** Network Theory (NeT) -- last of DS4's 5 fixed Step 1 families  
**Dataset:** DS4 -- Large Naturalistic OneSec Users (N=1,039)  
**Prerequisites:** None of the other 4 DS4 families -- only sequence engineering.  
**Blueprint:** `outputs/reports/stage4/step1_modeling/ds4_net_blueprint.md`

---

## Goal

Build a per-participant app-transition network from DS4's own event structure, using **attempt rows only** (`intervention_event`+`no_intervention`, excluding `app_close_event`) -- a verified, DS4-specific methodological decision, not a DS5 port (DS5 never had a close-tail event class to exclude).

## Not the goal

- No higher-order Markov structure -- first-order only, matching DS5's own FLAG M1.
- No session-boundary time-gap filter -- DS4 has no reliable pre-engineered session structure (unlike DS6's `group_id`, which memory already flags as unreliable).

## Core claim

> Pooling all 3 DS4 event classes into one sequence inflates rho_self because an `intervention_event` immediately followed by its own `app_close_event` (rho_self=0.959, n=348,028) is a logging artifact of that app's session ending, not a behavioral choice to stay on the same app. Excluding `app_close_event` gives a real, substantively different rho_self (~0.62) -- do not compare this in magnitude to DS5's ~0.92 without that caveat.

## Output

```
outputs/model_outputs/ds4_network_models/
  participant_network_metrics.csv
  transition_edge_list.csv
  ds4_net_summary.csv
  model_run_summary.md
  figures/
```


## FLAG decisions

| Flag | Decision |
|---|---|
| FLAG 1 (state space) | Top-20 apps by attempt-row frequency + 'other' (21 states) |
| FLAG 2 (event scope) | Attempt rows only, excluding app_close_event |
| FLAG 3 (eligibility) | MIN_TRANSITIONS=50 attempt-to-attempt pairs per participant |
| FLAG 4 (Markov order) | First-order |
| FLAG 5 (self-loops) | Included |
| FLAG 6 (time gaps) | No gap filter |


## DS4 structural cautions

- `app_close_event` is excluded here for a **different reason** than StoP/InT/DyS/StM excluded it from their own state-variable construction: those notebooks needed clean opened/dismissed outcomes (which `app_close_event` never has). NeT excludes it because it is a same-session close-tail, not a new engagement attempt -- verified via the rho_self class-pair breakdown in the blueprint.
- No experimental condition -- results describe naturalistic heterogeneity only.


## Theory

First-order Markov transition matrix: $T_{ij} = P(\text{app}_{t+1}=j \mid \text{app}_t=i)$. Stationary distribution $\pi$: left eigenvector of $T$ for $\lambda=1$, i.e. $\pi T = \pi$.

Transition (conditional) entropy: $H_{trans} = \sum_i p(i) H(\text{row}_i) = \sum_i p(i) \left(-\sum_j T_{ij}\log_2 T_{ij}\right) = H(Y\mid X)$

InT-NeT bridge (mutual information between consecutive apps):
$$I_{consec} = H_{app,attempts} - H_{trans} = H(Y) - H(Y\mid X) = I(X;Y) \geq 0$$

**Note:** $H_{app,attempts}$ must be the marginal entropy of the *current app* within the exact same population and state encoding $H_{trans}$ uses (attempts only, top-20+other) -- not InT's all-events `H_app`, which is a different population. Using the wrong marginal breaks the $H(Y)\geq H(Y|X)$ guarantee; see the implementation note in cell 12 for the real case this was caught on.


## Theory-to-data mapping

| Quantity | DS4 source |
|---|---|
| Sequence | `app_name`, attempt rows only, sorted by `timestamp_utc` per participant |
| T_ij | Row-normalized counts over top-20+other states |
| rho_self | Diagonal mass / total mass of the count matrix |
| H_trans | Conditional entropy H(current\|previous) |
| HHI_net | Herfindahl of attempt-row app-usage share, per participant |
| H_app_attempts | Marginal entropy of `app_current_enc`, attempts-only population -- the correct InT-NeT bridge partner |
| H_app_all_events | InT's `H_app`, joined for reference only -- NOT the bridge partner (different population, see cell 12) |


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

BEH_PATH = Path("outputs/cleaned/ds4_behavioral_clean.csv")
FEAT_PATH = Path("outputs/features/ds4/ds4_behavior_features.csv")
INT_PATH = Path("outputs/model_outputs/ds4_information_theory/participant_entropy_metrics.csv")
OUT_DIR = Path("outputs/model_outputs/ds4_network_models")
FIG_DIR = OUT_DIR / "figures"

TOP_K = 20
MIN_TRANSITIONS = 50

OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)


def shannon_entropy(counts):
    counts = np.asarray(counts, dtype=float)
    total = counts.sum()
    if total <= 0:
        return np.nan
    p = counts[counts > 0] / total
    return float(-(p * np.log2(p)).sum())


In [ ]:
beh = pd.read_csv(BEH_PATH, parse_dates=["timestamp_utc"], low_memory=False)

attempts = beh[beh["intervention_event_class"] != "app_close_event"].copy()
attempts = attempts.sort_values(["participant_id_canonical", "timestamp_utc"])
attempts["app_prev"] = attempts.groupby("participant_id_canonical")["app_name"].shift(1)
transitions = attempts.dropna(subset=["app_prev"]).copy()

print(f"beh: {len(beh):,} rows")
print(f"attempts (excl. app_close_event): {len(attempts):,} rows "
      f"({len(attempts)/len(beh):.1%})")
print(f"transitions: {len(transitions):,} rows across "
      f"{transitions['participant_id_canonical'].nunique()} participants")


In [ ]:
app_counts = attempts["app_name"].value_counts()
top_apps = app_counts.head(TOP_K).index.tolist()
coverage = app_counts.head(TOP_K).sum() / app_counts.sum()
print(f"top-{TOP_K} apps cover {coverage:.1%} of attempt rows")

top_apps_set = set(top_apps)
def encode_app(name):
    return name if name in top_apps_set else "other"

transitions["app_current_enc"] = transitions["app_name"].apply(encode_app)
transitions["app_prev_enc"] = transitions["app_prev"].apply(encode_app)

states = top_apps + ["other"]
print(f"state space size: {len(states)}")


In [ ]:
def conditional_entropy(ct):
    """ct: prev x current count DataFrame. Returns H(current|previous)."""
    total = ct.values.sum()
    if total <= 0:
        return np.nan
    row_sums = ct.sum(axis=1)
    h = 0.0
    for i in ct.index:
        n_i = row_sums[i]
        if n_i <= 0:
            continue
        p_i = n_i / total
        h += p_i * shannon_entropy(ct.loc[i].values)
    return h

records = []
transition_matrices = {}
count_matrices = {}
for pid, g in transitions.groupby("participant_id_canonical"):
    n = len(g)
    rec = dict(participant_id_canonical=pid, n_transitions=n, eligible=False,
               rho_self=np.nan, H_trans=np.nan)
    if n < MIN_TRANSITIONS:
        records.append(rec)
        continue
    ct = pd.crosstab(g["app_prev_enc"], g["app_current_enc"])
    ct = ct.reindex(index=states, columns=states, fill_value=0)
    row_sums = ct.sum(axis=1)
    T = ct.div(row_sums.replace(0, np.nan), axis=0)
    self_mask = np.eye(len(states), dtype=bool)
    rho_self = ct.values[self_mask].sum() / ct.values.sum()
    h_trans = conditional_entropy(ct)

    rec.update(eligible=True, rho_self=float(rho_self), H_trans=h_trans)
    records.append(rec)
    transition_matrices[pid] = T
    count_matrices[pid] = ct

participant_net = pd.DataFrame(records)
all_pids = beh["participant_id_canonical"].unique()
present = set(participant_net["participant_id_canonical"])
missing = [pid for pid in all_pids if pid not in present]
if missing:
    extra = pd.DataFrame([
        dict(participant_id_canonical=pid, n_transitions=0, eligible=False,
             rho_self=np.nan, H_trans=np.nan) for pid in missing
    ])
    participant_net = pd.concat([participant_net, extra], ignore_index=True)

n_eligible = int(participant_net["eligible"].sum())
print(f"Eligibility funnel: total={len(participant_net)}, "
      f"eligible (>=MIN_TRANSITIONS={MIN_TRANSITIONS})={n_eligible}")
print(f"rho_self median (eligible): "
      f"{participant_net.loc[participant_net['eligible'],'rho_self'].median():.4f}")


In [ ]:
def herfindahl(counts):
    counts = np.asarray(counts, dtype=float)
    total = counts.sum()
    if total <= 0:
        return np.nan
    shares = counts / total
    return float((shares ** 2).sum())

hhi_records = []
for pid, g in attempts.groupby("participant_id_canonical"):
    hhi_records.append(dict(participant_id_canonical=pid,
        HHI_net=herfindahl(g["app_name"].value_counts().values)))
hhi_df = pd.DataFrame(hhi_records)

participant_net = participant_net.merge(hhi_df, on="participant_id_canonical", how="left")

stage1_feats = pd.read_csv(FEAT_PATH)
check = participant_net.merge(
    stage1_feats[["participant_id_canonical", "app_herfindahl"]],
    on="participant_id_canonical", how="left",
)
r_hhi = check["HHI_net"].corr(check["app_herfindahl"])
print(f"corr(HHI_net [attempts-only], app_herfindahl [all events, Stage1]) = {r_hhi:.4f}")


In [ ]:
def stationary_distribution(T_df, tol=1e-10, max_iter=1000):
    """Power iteration on the participant's own visited state support."""
    T_full = T_df.fillna(0).values.astype(float)
    visited = T_full.sum(axis=1) > 0
    if visited.sum() < 1:
        return None, np.nan
    T = T_full[np.ix_(visited, visited)]
    row_sums = T.sum(axis=1, keepdims=True)
    if (row_sums <= 0).any():
        return None, np.nan
    T = T / row_sums
    n = T.shape[0]
    pi = np.ones(n) / n
    for _ in range(max_iter):
        pi_next = pi @ T
        if np.abs(pi_next - pi).sum() < tol:
            pi = pi_next
            break
        pi = pi_next
    if pi.sum() <= 0:
        return None, np.nan
    pi = pi / pi.sum()
    residual = float(np.max(np.abs(pi @ T - pi)))
    return pi, residual

pi_records = []
for pid, T in transition_matrices.items():
    pi, residual = stationary_distribution(T)
    ok = (pi is not None) and (residual < 1e-6)
    pi_records.append(dict(participant_id_canonical=pid,
                           is_stationary_eligible=bool(ok),
                           stationary_residual=residual,
                           H_pi=shannon_entropy(pi) if ok else np.nan))

pi_df = pd.DataFrame(pi_records)
participant_net = participant_net.merge(pi_df, on="participant_id_canonical", how="left")
print(f"H_pi computed for {int(pi_df['is_stationary_eligible'].sum())} stationary-eligible "
      f"participants of {len(pi_df)} eligible (residual < 1e-6 gate)")


In [ ]:
# NOTE (bug caught + fixed during implementation, see model_run_summary.md):
# InT's H_app is computed over ALL DS4 events, including app_close_event rows --
# a DIFFERENT population from H_trans's attempts-only, top-20-encoded population.
# For participants whose app_close_event rows are heavily concentrated on one app
# (dragging the all-events H_app down) while their attempt rows are more diverse,
# H_app < H_trans is possible even though H(Y) >= H(Y|X) always holds for the SAME
# population/encoding. Verified concretely on participant 034423D6...: 647
# app_close_event rows, 98% reddit; only 158 attempt rows, much more diverse --
# InT's H_app=0.86 bits is measuring the close-tail's low diversity, not the
# attempt population H_trans is built from.
#
# Fix: compute H_app_attempts -- the marginal entropy of app_current_enc within the
# SAME attempts-only, top-20-encoded population H_trans uses. This is the properly
# comparable bridge partner. InT's original H_app is kept alongside for reference.
int_metrics = pd.read_csv(INT_PATH)
participant_net = participant_net.merge(
    int_metrics[["participant_id_canonical", "H_app"]].rename(
        columns={"H_app": "H_app_all_events"}),
    on="participant_id_canonical", how="left",
)

happ_att_records = []
for pid, g in transitions.groupby("participant_id_canonical"):
    marginal_counts = g["app_current_enc"].value_counts()
    happ_att_records.append(dict(participant_id_canonical=pid,
        H_app_attempts=shannon_entropy(marginal_counts.values)))
happ_att_df = pd.DataFrame(happ_att_records)
participant_net = participant_net.merge(happ_att_df, on="participant_id_canonical", how="left")

participant_net["I_consec"] = participant_net["H_app_attempts"] - participant_net["H_trans"]

bridge_valid = participant_net.dropna(subset=["I_consec"])
n_negative = int((bridge_valid["I_consec"] < -1e-9).sum())
print(f"I_consec (using H_app_attempts, the corrected marginal) computed for "
      f"{len(bridge_valid)} participants; negative: {n_negative}")
print(bridge_valid["I_consec"].describe())

# Report how the ORIGINAL (buggy) version using InT's all-events H_app would have
# looked, for the record -- not exported as a primary column.
naive_I_consec = bridge_valid["H_app_all_events"] - bridge_valid["H_trans"]
n_negative_naive = int((naive_I_consec < -1e-9).sum())
print(f"\nFor comparison, the naive version (InT's all-events H_app, the bug): "
      f"{n_negative_naive}/{len(bridge_valid)} negative (min={naive_I_consec.min():.3f})")


In [ ]:
participant_net.to_csv(OUT_DIR / "participant_network_metrics.csv", index=False)
print(f"Wrote participant_network_metrics.csv ({len(participant_net)} rows)")

pooled_counts = sum(count_matrices.values())
edge_records = []
for i in states:
    for j in states:
        c = int(pooled_counts.loc[i, j])
        edge_records.append(dict(app_prev=i, app_current=j, count=c))
edge_df = pd.DataFrame(edge_records)
row_totals = edge_df.groupby("app_prev")["count"].transform("sum")
edge_df["probability"] = edge_df["count"] / row_totals.replace(0, np.nan)
edge_df.to_csv(OUT_DIR / "transition_edge_list.csv", index=False)
print(f"Wrote transition_edge_list.csv ({len(edge_df)} rows)")


In [ ]:
valid = participant_net[participant_net["eligible"]]
summary_row = {
    "n_total": len(participant_net),
    "n_eligible": int(participant_net["eligible"].sum()),
    "median_rho_self": float(valid["rho_self"].median()),
    "median_H_trans": float(valid["H_trans"].median()),
    "median_HHI_net": float(valid["HHI_net"].median()),
    "n_stationary_eligible": int(participant_net["is_stationary_eligible"].fillna(False).sum()),
    "median_H_pi": float(participant_net["H_pi"].median()),
    "r_HHInet_appherfindahl": float(r_hhi),
    "n_Iconsec_negative": n_negative,
    "n_Iconsec_negative_naive_buggy_version": n_negative_naive,
}
ds4_net_summary = pd.DataFrame([summary_row])
ds4_net_summary.to_csv(OUT_DIR / "ds4_net_summary.csv", index=False)
print(ds4_net_summary.T)

summary_lines = []
summary_lines.append("# DS4 NeT -- Model Run Summary")
summary_lines.append("")
summary_lines.append("**Notebook:** `54_ds4_network_models.ipynb`  ")
summary_lines.append("**Blueprint:** `outputs/reports/stage4/step1_modeling/ds4_net_blueprint.md`  ")
summary_lines.append("**Run date:** " + pd.Timestamp.now().strftime("%Y-%m-%d"))
summary_lines.append("")
summary_lines.append("## Summary")
summary_lines.append("")
summary_lines.append("```")
summary_lines.append(ds4_net_summary.T.to_string())
summary_lines.append("```")
summary_lines.append("")
summary_lines.append("## Note")
summary_lines.append("")
summary_lines.append("rho_self here excludes app_close_event (see blueprint SS1) -- "
                     "do not compare directly to DS5's rho_self without that caveat.")
summary_lines.append("")
summary_lines.append("## Bug caught and fixed during implementation")
summary_lines.append("")
summary_lines.append("Initial I_consec = InT's all-events H_app minus H_trans produced "
                     f"{n_negative_naive}/{len(bridge_valid)} negative values (min "
                     f"{naive_I_consec.min():.3f} bits) -- mathematically impossible "
                     "for a true H(Y)-H(Y|X). Root cause, verified concretely on "
                     "participant <participant-id>: InT's H_app is "
                     "computed over ALL events including app_close_event rows: 647 of "
                     "this participant's events were app_close_event, 98% reddit, "
                     "dragging their all-events H_app to 0.86 bits even though their "
                     "158 real attempt rows were much more diverse (H_trans=1.49 bits "
                     "on that same diverse population). Fixed by computing "
                     "H_app_attempts -- the current-app marginal within the exact same "
                     "attempts-only, top-20-encoded population H_trans uses -- instead "
                     "of reusing InT's differently-scoped H_app. "
                     f"Result: {n_negative}/{len(bridge_valid)} negative after the fix.")

with open(OUT_DIR / "model_run_summary.md", "w") as f:
    f.write("\n".join(summary_lines) + "\n")
print("Wrote model_run_summary.md")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.hist(valid["rho_self"], bins=40, color="#4C72B0", edgecolor="white")
ax.axvline(0.92, color="gray", linestyle="--", linewidth=1, label="DS5's rho_self (~0.92, not comparable)")
ax.set_xlabel("rho_self")
ax.set_ylabel("count")
ax.set_title(f"DS4 NeT: rho_self distribution (n={len(valid)}, attempts-only)")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig1_ds4_rho_self_distribution.png", dpi=150)
plt.close(fig)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.hist(valid["H_trans"], bins=40, color="#C44E52", edgecolor="white")
ax.set_xlabel("H_trans (bits)")
ax.set_ylabel("count")
ax.set_title("DS4 NeT: transition (conditional) entropy distribution")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig2_ds4_htrans_distribution.png", dpi=150)
plt.close(fig)


In [ ]:
pooled_T = pooled_counts.div(pooled_counts.sum(axis=1).replace(0, np.nan), axis=0)
fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(pooled_T.values, cmap="viridis", aspect="auto")
ax.set_xticks(range(len(states)))
ax.set_xticklabels(states, rotation=90, fontsize=7)
ax.set_yticks(range(len(states)))
ax.set_yticklabels(states, fontsize=7)
ax.set_xlabel("app_current")
ax.set_ylabel("app_prev")
ax.set_title("DS4 NeT: pooled transition matrix (attempts-only, top-20+other)")
fig.colorbar(im, ax=ax, label="P(current|prev)")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig3_ds4_transition_heatmap.png", dpi=150)
plt.close(fig)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(check["app_herfindahl"], check["HHI_net"], s=14, alpha=0.6, color="#55A868")
lims = [0, 1]
ax.plot(lims, lims, color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("app_herfindahl (Stage 1, all events)")
ax.set_ylabel("HHI_net (NeT, attempts-only)")
ax.set_title(f"Redundancy check (r={r_hhi:.3f})")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig4_ds4_hhi_redundancy_check.png", dpi=150)
plt.close(fig)


## Interpretation and limits

**Read the cells above for live numbers.**

- `rho_self` here is computed on attempt rows only, deliberately excluding `app_close_event`'s same-session close-tail (see blueprint SS1). Do not compare this number to DS5's `rho_self` (~0.92) without that construction difference in mind.
- **Bug caught and fixed (cell 12):** the naive `I_consec = InT's H_app - H_trans` produced negative values for 138/950 participants (min -0.63 bits) because InT's `H_app` is computed over ALL events (including `app_close_event`'s close-tail), a different population from `H_trans`'s attempts-only one. Fixed by computing `H_app_attempts` -- the marginal within the exact same population `H_trans` uses. See `model_run_summary.md` for the full root-cause writeup and the concrete participant it was diagnosed on.
- `HHI_net` (attempts-only) vs Stage 1's `app_herfindahl` (all events) -- reported correlation is a redundancy check, not an assumption either is dispensable.

### DS4 weakness flags

FLAG I (excludes ~47% of raw events, deliberately), FLAG II (no condition), FLAG III (first-order Markov unverified), FLAG IV (pi is a long-run approximation), FLAG V (rho_self not comparable to DS5 in magnitude) -- see `ds4_net_blueprint.md` SS7 for full text.


## Forward connections

| Connection | Mechanism | Status |
|---|---|---|
| DS4 NeT -> DS4 Step 2 (Phenotyping) | rho_self, HHI_net, H_trans as network-topology phenotype dimensions | Pending -- after all 5 DS4 Step 1 families (this closes them) |
| DS4 NeT -> Paper | Attempt-only sequence construction and the app_close_event-exclusion finding not yet in the conceptual paper | Before submission |

**This closes DS4 Step 1 -- all 5 fixed families (StoP/InT/DyS/StM/NeT) complete.**
